# 1. Revenue per customer is higher than the ledger

**Track:** debug-the-ai | **Difficulty:** easy | **Tags:** joins, row duplication, aggregation

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("001-orders-payments-join")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "orders": """
UEFSMRUEFeASFdIJTBWsAhUAEgAAsAkIiRMABQEAig0IAIsNCACMDQgAjQ0IAI4NCACPDQgAkA0IAJENCACSDQgAkw0IAJQNCACV
DQgAlg0IAJcNCACYDQgAmQ0IAJoNCACbDQgAnA0IAJ0NCACeDQgAnw0IAKANCAChDQgAog0IAKMNCACkDQgApQ0IAKYNCACnDQgA
qA0IAKkNCACqDQgAqw0IAKwNCACtDQgArg0IAK8NCACwDQgAsQ0IALINCACzDQgAtA0IALUNCAC2DQgAtw0IALgNCAC5DQgAug0I
ALsNCAC8DQgAvQ0IAL4NCAC/DQgAwA0IAMENCADCDQgAww0IAMQNCADFDQgAxg0IAMcNCADIDQgAyQ0IAMoNCADLDQgAzA0IAM0N
CADODQgAzw0IANANCADRDQgA0g0IANMNCADUDQgA1Q0IANYNCADXDQgA2A0IANkNCADaDQgA2w0IANwNCADdDQgI3hMABQEA3w0I
AOANCADhDQgA4g0IAOMNCADkDQgA5Q0IAOYNCADnDQgA6A0IAOkNCADqDQgA6w0IAOwNCADtDQgA7g0IAO8NCADwDQgA8Q0IAPIN
CADzDQgA9A0IAPUNCAD2DQgA9w0IAPgNCAD5DQgA+g0IAPsNCAD8DQgA/Q0IAP4NCAD/DQgEABQlDwQAAQ0IAAINCAADDQgABA0I
AAUNCAAGDQgABw0IAAgNCAAJDQgACg0IAAsNCAAMDQgADQ0IAA4NCAAPDQgAEA0IABENCAASDQgAEw0IABQNCAAVDQgAFg0IABcN
CAAYDQgAGQ0IABoNCAAbDQgAHA0IPB0UAAAAAAAAHhQAAAAAAAAVABXCAhXKAiwVrAIVEBUGFQYcGAgeFAAAAAAAABgIiRMAAAAA
AAAWACgIHhQAAAAAAAAYCIkTAAAAAAAAEREAAAChAfCgAwAAAKwCAQgnAAECAwQFBgcICQoLDA0ODxAREhMUFRYXGBkaGxwdHh8g
ISIjJCUmJygpKissLS4vMDEyMzQ1Njc4OTo7PD0+P0BBQkNERUZHSElKS0xNTk9QUVJTVFVWV1hZWltcXV5fYGFiY2RlZmdoaWpr
bG1ub3BxcnN0dXZ3eHl6e3x9fn+AgYKDhIWGh4iJiouMjY6PkJGSk5SVAAAVBBXwBBXaAkwVThUAEgAAuAIEiQAJAQCACQcEAIgN
CABvDQgAcQ0IAG4NCARpAAkBAHcJBwQAgQ0IAHMNCACLDQgAdQ0IAGgNCABsDQgAfQ0IAHoNCAByDQgAaw0IAIUNCAB7DQgAZg0I
AIoNCABnDQgAfA0IAGoNCAB2DQgAZQ0IAG0NCACEDQgAeQ0IAIYNCAR0AAkBAIIJBwQAfg0IAIcNCAB/DQgAjA0IPHAAAAAAAAAA
eAAAAAAAAAAVABX2ARX8ASwVrAIVEBUGFQYcGAiMAAAAAAAAABgIZQAAAAAAAAAWACgIjAAAAAAAAAAYCGUAAAAAAAAAEREAAAB7
8HoDAAAArAIBBidAIAxEYRwBkijDwiyNwzxQBEiTE1GVdWGZoAXGtnAS1kWOFGiWBXnRM2wXMUiY8y2RszTO8VGVISSZBEqLdmHZ
tBxNKIoJVylX9YhZJmJFkSQQJoQGMYVZV3wDOCRJ02SgAEKUOIokBHZgkGRdU5rWBgAVBBXgBBXqBEwVmAEVABIAALAC9C8BkE4A
AMNOAADOTgAAs04AAIlOAAC+TgAAzU4AAJtOAACETgAAo04AAMBOAACgTgAAl04AAI9OAACVTgAAfE4AAJxOAAChTgAAmU4AAKdO
AACsTgAAx04AAJ1OAACOTgAA0k4AAIZOAAB/TgAAik4AALFOAACNTgAAu04AAIJOAACUTgAArk4AAMJOAACqTgAAt04AALROAACD
TgAApE4AAMFOAAB6TgAAjE4AAMROAACyTgAAok4AAK1OAACeTgAAr04AAIhOAACRTgAAvU4AALVOAAC2TgAAk04AANFOAADJTgAA
y04AALBOAADPTgAAuk4AAMhOAAC/TgAAgU4AAHlOAAC8TgAAfk4AAItOAACWTgAAyk4AAKhOAACfTgAAzE4AAMZOAACpTgAApk4A
ABUAFZwCFaQCLBWsAhUQFQYVBhwYBNJOAAAYBHlOAAAWACgE0k4AABgEeU4AABERAAAAjgHwjQMAAACsAgEHJ4CAQDAgFAwHREJR
WCgYDccD0ohIJglKxXJhPCqYjGbD6Xg+BVBINCKVTJpT4UG4fFAp1YrVgrhesEVsJZthaDWVomJj3Cy43AWj2wVoiQGvZ1nEQr4f
UDVgRAjBUwEhHFaIluJIlCLuAqHgC2F8FI8jJKcQI6wgqVWIJawllZgT7EdYAgAVBBXSCxXcC0wVqgIVABIAAOkF9OgCAAADurAA
AB4HgAAAJB0eAAAglAoAAB9chAAAEcDsAAAjewIAAAV7hAAADmP8AAAgBN8AAAbYJgAAAMksAAACGt4AABKI7AAAFJ9MAAAAwuwA
ACUhqgAACkhrAAAhTcgAABHcpwAAHVUuAAAWH0gAAAKGFwAAFd4MAAAfdhoAACMqJgAAHMJOAAAK0FsAACE+8AAAIfGmAAACnSQA
AAPyJwAAE4D3AAAVj4gAABwsgAAAF4kyAAAe7+sAACV83wAAI/siAAAEVPwAAAOD/wAAA2qcAAATUt8AABVBBAAAAcGbAAAGr1QA
ACBxEwAADU/PAAATQqoAAALoVgAADKqUAAAX+xAAAB6Y0AAAHehAAAACW1IAABVOrwAAFhP0AAANuRYAAAx/NwAACWV3AAAeHGYA
ABi7QAAAC2JzAAAlmZYAAAzfUAAAJXJUAAAaiSoAAA8diAAABqHaAAAGniMAABg9dwAABgHkAAAHw04AABUcsAAAJU7IAAAfk/wA
AAj+WAAAHf7nAAAjdCsAAB3K9AAAB8D2AAAPrwoAABRt4gAAJQfiAAANx1gAABYZawAAEbOjAAAU4t8AAAS04gAAAFyTAAAmIIsA
ABY17wAAG9ggAAAgsFwAABht5wAAB4lLAAAOHg8AAA5yPgAAFqG/AAAQZHwAABzk3wAAJYonAAAZazoAABcmLAAAH3d3AAACGRwA
ABYAbAAAEIWwAAAMg4MAABNuOAAAF1xGAAAiLsgAABWEygAAGHaAAAAPGQoAAByZfAAAHqg+AAAmFNQAABKLqAAACRXIAAAZZJYA
ACWhygAAFAggAAAh8goAAB9IAgAAF4SCAAAY7zQAABb8LgAABUFPAAAWiYgAACCLcgAAFwoMAAAciEwAAAsZzAAAB4mwAAASNuQA
ABB94AAAGptUAAAZfpAAACSqhgAAFGK/AAAHuCwAAB4sagAABgYwAAAW+s8AAA/t7gAABN1QAAAcHzgAACGSIxUAFcICFcoCLBWs
AhUQFQYVBhwYBQAAJiCLGAUAAABckxYAKAUAACYgixgFAAAAXJMREQAAAKEB8KADAAAArAIBCCcAAQIDBAUGBwgJCgsMDQ4PEBES
ExQVFhcYGRobHB0eHyAhIiMkJQ8mJygpKissLS4vMDEyMzQ1Njc4OTo7PD0+P0BBQkNERUZHSElKS0xNTk9QUVJTVFVWV1hZWltc
XV5fYGFiY2RlZmdoaWprbG1ub3BxcnN0dXZ3eHl6e3x9fn+AgYKDhIWGh4iJiouMjY6PkJGSk5QAABUEGVw1ABgGc2NoZW1hFQgA
FQQlAhgIb3JkZXJfaWQAFQQlAhgLY3VzdG9tZXJfaWQAFQIlAhgKb3JkZXJfZGF0ZSUMTGwAAAAVDhUKFQIYDG9yZGVyX2Ftb3Vu
dCUKFQQVFCxcFQQVFAAAABasAhkcGUwmABwVBBk1AAYQGRgIb3JkZXJfaWQVAhasAhbIFhbCDSb8CSYIHBgIHhQAAAAAAAAYCIkT
AAAAAAAAFgAoCB4UAAAAAAAAGAiJEwAAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgCsAgAAACYAHBUEGTUABhAZGAtjdXN0
b21lcl9pZBUCFqwCFooIFvoFJsQQJsoNHBgIjAAAAAAAAAAYCGUAAAAAAAAAFgAoCIwAAAAAAAAAGAhlAAAAAAAAABERABksFQQV
ABUCABUAFRAVAgA8KQYZJgCsAgAAACYAHBUCGTUABhAZGApvcmRlcl9kYXRlFQIWrAIWgggWlAgm0BgmxBMcGATSTgAAGAR5TgAA
FgAoBNJOAAAYBHlOAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYArAIAAAAmABwVDhk1AAYQGRgMb3JkZXJfYW1vdW50FQIWrAIW
og8WtA8m1icm2BscGAUAACYgixgFAAAAXJMWACgFAAAmIIsYBQAAAFyTEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAKwCAAAAFvY1
FqwCJggWhCsAGRwYDEFSUk9XOnNjaGVtYRisAy8vLy8vemdCQUFBUUFBQUFBQUFLQUF3QUJnQUZBQWdBQ2dBQUFBQUJCQUFNQUFB
QUNBQUlBQUFBQkFBSUFBQUFCQUFBQUFRQUFBRE1BQUFBaEFBQUFFZ0FBQUFFQUFBQVZQLy8vd0FBQVFjUUFBQUFLQUFBQUFRQUFB
QUFBQUFBREFBQUFHOXlaR1Z5WDJGdGIzVnVkQUFBQUFBSUFBd0FCQUFJQUFnQUFBQUtBQUFBQWdBQUFKVC8vLzhBQUFFSUVBQUFB
Q1FBQUFBRUFBQUFBQUFBQUFvQUFBQnZjbVJsY2w5a1lYUmxBQUFBQUFZQUNBQUdBQVlBQUFBQUFBQUF6UC8vL3dBQUFRSVFBQUFB
SEFBQUFBUUFBQUFBQUFBQUN3QUFBR04xYzNSdmJXVnlYMmxrQUx6Ly8vOEFBQUFCUUFBQUFCQUFGQUFJQUFZQUJ3QU1BQUFBRUFB
UUFBQUFBQUFCQWhBQUFBQWtBQUFBQkFBQUFBQUFBQUFJQUFBQWIzSmtaWEpmYVdRQUFBQUFDQUFNQUFnQUJ3QUlBQUFBQUFBQUFV
QUFBQUE9ABggcGFycXVldC1jcHAtYXJyb3cgdmVyc2lvbiAyNS4wLjEZTBwAABwAABwAABwAAAAYBAAAUEFSMQ==
""",
    "payments": """
UEFSMRUEFZAYFaYMTBWCAxUAEgAAiAwMgTgBAAEBAIINCACDDQgAhA0IAIUNCACGDQgAhw0IAIgNCACJDQgAig0IAIsNCACMDQgA
jQ0IAI4NCACPDQgAkA0IAJENCACSDQgAkw0IAJQNCACVDQgAlg0IAJcNCACYDQgAmQ0IAJoNCACbDQgAnA0IAJ0NCACeDQgAnw0I
AKANCAChDQgAog0IAKMNCACkDQgApQ0IAKYNCACnDQgAqA0IAKkNCACqDQgAqw0IAKwNCACtDQgArg0IAK8NCACwDQgAsQ0IALIN
CACzDQgAtA0IALUNCAC2DQgAtw0IALgNCAC5DQgAug0IALsNCAC8DQgAvQ0IAL4NCAC/DQgAwA0IAMENCADCDQgAww0IAMQNCADF
DQgAxg0IAMcNCADIDQgAyQ0IAMoNCADLDQgAzA0IAM0NCADODQgAzw0IANANCADRDQgA0g0IANMNCADUDQgA1Q0IANYNCADXDQgA
2A0IANkNCADaDQgA2w0IANwNCADdDQgA3g0IAN8NCADgDQgA4Q0IAOINCADjDQgA5A0IAOUNCADmDQgA5w0IAOgNCADpDQgA6g0I
AOsNCADsDQgA7Q0IAO4NCADvDQgA8A0IAPENCADyDQgA8w0IAPQNCAD1DQgA9g0IAPcNCAD4DQgA+Q0IAPoNCAD7DQgA/A0IAP0N
CAD+DQgA/w0IBAA5afgAAQ0IAAINCAADDQgABA0IAAUNCAAGDQgABw0IAAgNCAAJDQgACg0IAAsNCAAMDQgADQ0IAA4NCAAPDQgA
EA0IABENCAASDQgAEw0IABQNCAAVDQgAFg0IABcNCAAYDQgAGQ0IABoNCAAbDQgAHA0IAB0NCAAeDQgAHw0IACANCAAhDQgAIg0I
ACMNCAAkDQgAJQ0IACYNCAAnDQgAKA0IACkNCAAqDQgAKw0IACwNCAAtDQgALg0IAC8NCAAwDQgAMQ0IADINCAAzDQgANA0IADUN
CAA2DQgANw0IADgNCAA5DQgAOg0IADsNCAA8DQgAPQ0IAD4NCAA/DQg8QDkBAAAAAABBOQEAAAAAABUAFaIDFaoDLBWCAxUQFQYV
BhwYCEE5AQAAAAAAGAiBOAEAAAAAABYAKAhBOQEAAAAAABgIgTgBAAAAAAAREQAAANEB8NADAAAAggMBCDMAAQIDBAUGBwgJCgsM
DQ4PEBESExQVFhcYGRobHB0eHyAhIiMkJSYnKCkqKywtLi8wMTIzNDU2Nzg5Ojs8PT4/QEFCQ0RFRkdISUpLTE1OT1BRUlNUVVZX
WFlaW1xdXl9gYWJjZGVmZ2hpamtsbW5vcHFyc3R1dnd4eXp7fH1+f4CBgoOEhYaHiImKi4yNjo+QkZKTlJWWl5iZmpucnZ6foKGi
o6SlpqeoqaqrrK2ur7CxsrO0tba3uLm6u7y9vr/AAAAAAAAAABUEFeARFZYJTBWcAhUAEgAA8AgI4BMABQEAog0IAO8NCACKDQgA
wA0IAI4NCAQQFAUvBAC4DRAAEw0QANANEADsDQgA9A0IAL0NCADXDQgACA0wAM8NEACqDQgA5A0IAP8NCAC1DQgAEg0wALINEADh
DQgA8Q0IAJUNCADRDQgA4w0IAOYNCAD2DQgAmQ0IAJ4NCADEDQgA6A0IAOcNCACpDQgAAA14ABYNCAAPDQgA2g0gAMcNCACoDQgA
AQ0gALkNEADiDQgAxg0IAPINCACLDQgAjw0IALANCAAXDUAAwg0QAL8NCACuDQgAlg0IANgNCAC6DQgAAw04AKENEACaDQgAHg0Y
AJ8NEAAVDRAA/Q0QAJANCAAHDRgA+w0QAAYNEADbDRAAyA0IAJINCADcDQgAsw0IALENCADBDQgA9w0IALcNCADWDQgA5Q0IAKsN
CAAYDWgAvA0QAPANCADODQgADQ0gAK8NEACUDQgA0w0IALsNCACTDQgApg0IAKQNCAAFDUAA1Q0QAB0NEACbDRAAiQ0IAMMNCADu
DQgAjA0IAL4NCADrDQgACQ1AAJwNEAjeEwAFAQDLDQgAGQ0gAAINCAD1DRgA6g0IAN0NCAAKDSAAkQ0QAP4NCAQqIwVPBACYDRAA
Gg0oAKUNEADKDQgAtA0IAPMNCAAMDSgA2Q0QAPoNCACjDQgAHA0gAMkNEADFDQgArQ0IACkNeACNDRAAGw0wAA4NCAAEDQgA+Q0g
APwNCACdDQgA0g0IAPgNCADNDQgAtg0IPOkTAAAAAAAACxQAAAAAAAAVABWiAxWqAywVggMVEBUGFQYcGAgqIwAAAAAAABgIiRMA
AAAAAAAWACgIKiMAAAAAAAAYCIkTAAAAAAAAEREAAADRAfDQAwAAAIIDAQgzAAECAwQFBgcICQoLDA0ODxAREhMUFRYXGBkaGxwd
HgEfICEiIyQlJicoJBwpKissLS4vMDEyMzQ1Njc4OTo7DDw9Pj9AQUJDRB9FRkdILEdJSgJLA0xNEi9OG09QUVJTVFUdF1ZXLVhZ
WjlWW1xdWl5fOGBIYWIGRGM3MmRlCGZnaGlqa0YLbG0WbjVvcHFvciduc3R1EW1WdncdeEAYeXp7U198fX5jfwVjgDiBgmWDhFuF
Z12GBIeIiYpmi4yNWgAAAAAAAAAVBBWAGBWAFUwVgAMVABIAAIAMkACY9MVZLwYAAErYZqsuBgAA+Bh3yy0GAAD6vKV7MAYAADnj
OFYBCCxvd7HqKwYAAOZlOmQBIAxTOwtxARAM+gk52QE4CJdCAQVILL5lZCcsBgAANbPohAEgDETJQ+YBCCymbdqOMQYAAPW8ZaUB
QAxfm9OfARAMCS40rgEgDDUlDNsBaAwDON5jARgMLNLkfgFYDL50xIYBGAx4XS/IAVgM/CZN8gFADMRsBe4BuAxBYEQRASAMGFtx
bQEQDEsli78BCAxkXyizASgMc/7zDQE4DIOqxWEBCAzNeDDOARgMJ6z3pQFgDHmnxCEBMAza5fDOARAMcSoy0wGQDKy1xKEBiAyt
GyF2AWAMupjlcwEoDGzuIogBEAzq3twTASAM5dpbvgE4DIbdmsoBGAz1w7xyASgMX7l3EAFwDKOCLNMBIAwi1XogAVgMpZyxMQEg
DCrT+2kBiAz7nv0GASAMHrjZVAFQDF710RwBSAzsDCBbATAMYuMYtgEQDFfaWiMBUAx34aJaARgMlnAmNwEQDKyG1bQBSAw01dF4
AQgM2avmBQFQDCZRrIoBEAxFh1L4AQgMCqXN1wEwDPejQSsBIAz1WPDpAUgMPES2LAEgDOTiTAYBECzvZ3+8KgYAAKZMmTwBkAyV
nwsBAQgMiikrgwE4DPmTyuABIAwkhWxOAVAMUNTokQFADKddWSEB2AyGADJ/AUgMWra2pgE4DE00eNEBOAwAF/gUAcgMwhzBaAEw
DC5l1tkBGAwp4ZG6AVAIvnm3JUAM3NiP0gEgDBL3OEwBUAymS+d3ATgMWfCeAgEYCF5K5wXoDF+yLmoBYAjWoeIFWAxPUrcbAXgI
F8zKJRAMs8IGbQFgDA890N8BYAj5mW8F8AwsuPr5AVgMXolTzwEwDJpG59QBEAzm66I2AWgM4qWpXwHYDB08TecBGAwumuM7ARgM
dTRODgEIDF7phwsBWAzHoH70AZAM+Ame/wEwDKqyE3oBEAxQEXjyAQgMpyIuSgEwDK9XElgBMAwwhXmtASgM5KIu6wF4DI3AZ4AB
KAwWlJ3hAaAMFiu8dQEgDJsR5JUBGAwzoEGHAUAMwZx6DwEICGcXpiUADJH4OVAB+Ayp1tljASgME1OvmAEIDI3fREkBQCwHcKIM
LgYAAOK8BCoBEAxijJGpAWAMg1nxhAEYCJjq3yWQDFEyw2cBOAzdmAJuAVAMVLHy1QGYDOr26aIBEAx8vsToIQAMhLw3CwEYDB+L
36cBGAwanjKJARAMXKJ3SQEIDO5+V/cBWAxoLC76AVAMAXJDMAEoDJKcULYBEAzNAwVRAQgM2QJg6wGQDDmRKyEBMAzbsZ9UARgI
dT6tBdgIYzqcZUgMnXX0KgFADPLjpgMBKCxsJYZMLAYAAC4nlbQBMAy3PjjXIRAMARIP9AEgDDYqCUIBEAzjsNJiAQgMlUpvEQEI
DGudNmEBOAzMiWMAAVAILOJvJUgMZhtk8gEgDP4HCwMBGAzrb2DCARAMuJ8a1AHQDA635H0BOAxZBZZDARAMm6MP5gFoDENywN0B
gAzu02xTASAMgjimDyEgDAEcbugBKAxNSL1NARgMydFeHQEwDH4D6o0BMAxhPRczASgMLyXC4QEYDKq4uIIBCAisBhJFCAw9rlOA
AUAMP6ZsWwEYCFKHaUVICJJbB0XgDFY/8ewBIAyen3NjAagMiMz4TgEIDA8xDWIBcAi9LAyFGAxKZsroARAMnhNmiAEIDJCeGWYB
gAw9jkuGAYAMI1+1KAE4OO4hOQkvBgAACvV2gy0GABUAFaIDFaoDLBWCAxUQFQYVBhwYCAAfi9+nMQYAGAgAYoyRqSoGABYAKAgA
H4vfpzEGABgIAGKMkakqBgAREQAAANEB8NADAAAAggMBCDMAAQIDBAUGBwgJCgsMDQ4PEBESExQVFhcYGRobHB0eHyAhIiMkJSYn
KCkqKywtLi8wMTIzNDU2Nzg5Ojs8PT4/QEFCQ0RFRkdISUpLTE1OT1BRUlNUVVZXWFlaW1xdXl9gYWJjZGVmZ2hpamtsbW5vcHFy
c3R1dnd4eXp7fH1+f4CBgoOEhYaHiImKi4yNjo+QkZKTlJWWl5iZmpsYnJ2en6ChoqOkpaanqKmqq6ytrq+wsbKztLW2t7i5uru8
vb6/AAAAAAAAABUEFYYNFZANTBXOAhUAEgAAwwb0QgMAABGzowAAIyomAAAYZhgAAB4HgAAAAltSAAARwOwAAAeJsAAAIHETAAAL
XesAABg9dwAAFqG/AAADN1sAABf7EAAAHf7nAAAY7zQAAAaeIwAAFY+IAAAmIIsAACYU1AAAFUEEAAAQfeAAAAOD/wAAFOLfAAAK
B6AAAAIa3gAABgHkAAAAXJMAABvYIAAADIODAAAKyCcAABYfSAAADH83AAAYbecAACCwXAAAE4D3AAASi6gAAASydgAACxnMAAAH
wPYAABSrPQAAA/InAAAPsEkAAAkVyAAADU/PAAAEtOIAAB4cZgAAH3d3AAAkHR4AACN7AgAAI/siAAAHuCwAABTQjQAAHehAAAAl
fN8AABKI7AAAI3QrAAANfXYAACWhygAAG3krAAAKSGsAACGSIwAAAoYXAAAkqoYAAByZfAAABXuEAAAXhIIAABh2gAAAH0gCAAAP
rwoAAATmOQAAIATfAAAUbeIAAANqnAAABFT8AAAVTq8AABNuOAAADSQPAAAGr1QAAAj+WAAAFjXvAAAcLIAAAB4sagAADKqUAAAZ
azoAAAah2gAAFIjYAAAAwuwAAADJLAAAD+H5AAANHowAAAwsDwAAAuhWAAAG2CYAACHxpgAACbTjAAAD/O8AAAayEgAAIfIKAAAf
k/wAABwfOAAAAMTKAAAhTcgAAADw2gAAB6ilAAANuRYAABzk3wAAIJQKAAAGfDoAAAU9kgAABcU0AAABQ2cAAA5yPgAAALKVAAAP
PWkAAACz+QAACvnDAAAlclQAAAYGMAAAGWSWAAAQhbAAABLJEQAADh4PAAAlB+IAAAVBTwAAA0HwAAAeqD4AAAalLAAACyIMAAAE
EAMAABb6zwAAIT7wAAAM31AAAAI+jwAAE1LfAAACGRwAAAp3igAAIItyAAAdyvQAABWEygAAHMJOAAACgTQAAALJ1gAABN1QAAAl
mZYAAAlldwAAFKTIAAAe7+sAAAS2dgAABLK8AAAd+SUAAB9chAAAD+3uAAAWSZkAAByITAAAFAggAAAiLsgAAALNlQAADxkKAAAd
VS4AAAfDTgAAF1xGAAAPHYgAABEorgAAAcGbAAAHiUsAABaJiAAAAFauFQAVogMVqgMsFYIDFRAVBhUGHBgFAAAmIIsYBQAAAFau
FgAoBQAAJiCLGAUAAABWrhERAAAA0QHw0AMAAACCAwEIMwABAgMEBQYHCAkKCwwNDg8QERITFBUWFxgZGhscHR4BHyAhIiMkJSYn
KCkcKissLS4vMDEyMzQ1Njc4OTo7PAw9Pj9AQUJDREUfRkdISS1ISktMTQNOTxIwUBtRUlNUVVZXWFlaWy5cXV5fYGFiY2RlZmdo
SWlqBmtsbW5vcHFyc3R1dndHeHl6Fns2fH1+f1aAe4GCgxF6hIWGh4hBGImKi4yNjo+QkZIFk5SVlpeYmZphm5xjnQSen6ChoqOk
paYAAAAAAAAAFQQVThVSTBUIFQASAAAnmAQAAABjYXJkCgAAAG5ldGJhbmtpbmcGAAAAd2FsbGV0AwAAAHVwaRUAFXYVeiwVggMV
EBUGFQYcNgAoBndhbGxldBgEY2FyZBERAAAAO+gDAAAAggMBAjMACSdGkmgfiNr18iVSeTRRHHe1wvLJ69zCrdiTX/y9WqR+T8Z/
x4UgBBXmpmja7UMCABUEFUAVREwVBhUAEgAAIHwHAAAAU1VDQ0VTUwYAAABGQUlMRUQHAAAAUEVORElORxUAFXoVgAEsFYIDFRAV
BhUGHDYAKAdTVUNDRVNTGAZGQUlMRUQREQAAAD3wPAMAAACCAwECCwAEAAEQAEBBBEAQAB0BAEEQICBUEIABCQUIAAARQEAAAUAA
QAAQUAAUEAAJBQAAQABAAAAVBBl8NQAYBnNjaGVtYRUMABUEJQIYCnBheW1lbnRfaWQAFQQlAhgIb3JkZXJfaWQAFQQlAhgHcGFp
ZF9hdCUUTIwSHCwAAAAAABUOFQoVAhgGYW1vdW50JQoVBBUULFwVBBUUAAAAFQwlAhgGbWV0aG9kJQBMHAAAABUMJQIYBnN0YXR1
cyUATBwAAAAWggMZHBlsJgAcFQQZNQAGEBkYCnBheW1lbnRfaWQVAhaCAxbYHBb2ECbQDCYIHBgIQTkBAAAAAAAYCIE4AQAAAAAA
FgAoCEE5AQAAAAAAGAiBOAEAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgCCAwAAACYAHBUEGTUABhAZGAhvcmRlcl9pZBUC
FoIDFqgWFuYNJrYaJv4QHBgIKiMAAAAAAAAYCIkTAAAAAAAAFgAoCCojAAAAAAAAGAiJEwAAAAAAABERABksFQQVABUCABUAFRAV
AgA8KQYZJgCCAwAAACYAHBUEGTUABhAZGAdwYWlkX2F0FQIWggMWyBwW0BkmhjQm5B4cGAgAH4vfpzEGABgIAGKMkakqBgAWACgI
AB+L36cxBgAYCABijJGpKgYAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAIIDAAAAJgAcFQ4ZNQAGEBkYBmFtb3VudBUCFoIDFrYR
FsgRJuZFJrQ4HBgFAAAmIIsYBQAAAFauFgAoBQAAJiCLGAUAAABWrhERABksFQQVABUCABUAFRAVAgA8KQYZJgCCAwAAACYAHBUM
GTUABhAZGAZtZXRob2QVAhaCAxasAha0AibqSib8SRw2ACgGd2FsbGV0GARjYXJkEREAGSwVBBUAFQIAFQAVEBUCADwW3BEZBhkm
AIIDAAAAJgAcFQwZNQAGEBkYBnN0YXR1cxUCFoIDFqoCFrQCJpBNJrBMHDYAKAdTVUNDRVNTGAZGQUlMRUQREQAZLBUEFQAVAgAV
ABUQFQIAPBbGFBkGGSYAggMAAAAW1GUWggMmCBbcTgAZHBgMQVJST1c6c2NoZW1hGJgELy8vLy80Z0JBQUFRQUFBQUFBQUtBQXdB
QmdBRkFBZ0FDZ0FBQUFBQkJBQU1BQUFBQ0FBSUFBQUFCQUFJQUFBQUJBQUFBQVlBQUFBY0FRQUExQUFBQUp3QUFBQmdBQUFBTUFB
QUFBUUFBQUFNLy8vL0FBQUJCUkFBQUFBWUFBQUFCQUFBQUFBQUFBQUdBQUFBYzNSaGRIVnpBQURZLy8vL05QLy8vd0FBQVFVUUFB
QUFIQUFBQUFRQUFBQUFBQUFBQmdBQUFHMWxkR2h2WkFBQUJBQUVBQVFBQUFCZy8vLy9BQUFCQnhBQUFBQWdBQUFBQkFBQUFBQUFB
QUFHQUFBQVlXMXZkVzUwQUFBSUFBd0FCQUFJQUFnQUFBQUtBQUFBQWdBQUFKai8vLzhBQUFFS0VBQUFBQ0FBQUFBRUFBQUFBQUFB
QUFjQUFBQndZV2xrWDJGMEFBQUFCZ0FJQUFZQUJnQUFBQUFBQWdETS8vLy9BQUFCQWhBQUFBQWNBQUFBQkFBQUFBQUFBQUFJQUFB
QWIzSmtaWEpmYVdRQUFBQUF2UC8vL3dBQUFBRkFBQUFBRUFBVUFBZ0FCZ0FIQUF3QUFBQVFBQkFBQUFBQUFBRUNFQUFBQUNRQUFB
QUVBQUFBQUFBQUFBb0FBQUJ3WVhsdFpXNTBYMmxrQUFBSUFBd0FDQUFIQUFnQUFBQUFBQUFCUUFBQUFBPT0AGCBwYXJxdWV0LWNw
cC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRlsHAAAHAAAHAAAHAAAHAAAHAAAAGAFAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "001-orders-payments-join"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

orders = spark.read.parquet((_data_dir / "orders.parquet").resolve().as_posix())
orders.createOrReplaceTempView("orders")
payments = spark.read.parquet((_data_dir / "payments.parquet").resolve().as_posix())
payments.createOrReplaceTempView("payments")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'customer_id': '002d7e3b5cffbb1f1a37ab674e245518240b0ce849ff33c3f92f36c78119e86b',
                         'paid_orders': '32e5fcfb0f902e335d25e1dfc1cdca80446275b9a68eb2c34753a603ea554c24',
                         'revenue': '0b75d7dcfe394f02706509d7d5770002f7ba793bcd0c95bbbfb9ee6b4fb6208c'},
 'columns': [['customer_id', 'bigint'], ['paid_orders', 'bigint'], ['revenue', 'decimal(12,2)']],
 'fingerprint': '50ad6dfc372de3bc45db76a201903ca4f6178fda8d3a09a41c672c5faf45c649',
 'order_matters': False,
 'row_count': 39}

check = make_check(EXPECTED)

## The situation

A teammate asked an AI assistant for a "revenue per customer" report and shipped what
it wrote. The job runs without errors and the output looks reasonable. Finance has now
compared it to the ledger: the report's total revenue is too high, and only some
customers are off.

The AI-written code is in the answer cell. Find the bug and fix it.

## Input tables

### `orders`

One row per order.

| Column | Type | Description |
|---|---|---|
| `order_id` | BIGINT | Unique per order. |
| `customer_id` | BIGINT | The customer who placed the order. |
| `order_date` | DATE | Date the order was placed. |
| `order_amount` | DECIMAL(10,2) | Full value of the order in rupees. |

### `payments`

One row per payment attempt. An order can have no rows here, one row, or several:
a failed attempt followed by a retry, an order paid in two or three parts, or
occasionally an order the gateway charged twice in full.

| Column | Type | Description |
|---|---|---|
| `payment_id` | BIGINT | Unique per payment attempt. |
| `order_id` | BIGINT | The order the attempt was for. A few rows point to orders that are not in `orders`. |
| `paid_at` | TIMESTAMP_NTZ | When the attempt happened. |
| `amount` | DECIMAL(10,2) | Amount of this attempt in rupees. |
| `method` | STRING | `upi`, `card`, `netbanking` or `wallet`. |
| `status` | STRING | `SUCCESS`, `FAILED` or `PENDING`. |

## Requirement

- An order is **paid** if it has at least one `SUCCESS` payment.
- A customer's **revenue** is the sum of `order_amount` over their paid orders.
  Each paid order counts once, for its `order_amount`, however many payments it
  took and whatever those payments add up to.
- Every customer in `orders` must appear in the result. A customer with no paid
  orders has `paid_orders` = 0 and `revenue` = 0.00.

## Expected output

One row per customer.

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | |
| `paid_orders` | BIGINT | Number of paid orders. |
| `revenue` | DECIMAL(12,2) | Sum of `order_amount` over paid orders. |

In [ ]:
orders.show(5)
payments.show(5)

## Your answer

The cell below is the AI-written code. It runs, but its output is wrong. Find the bug and fix it. The tables are available as DataFrames and as views: `orders`, `payments`.

In [ ]:
from pyspark.sql import functions as F


def solve(spark):
    """Revenue per customer from paid orders."""
    orders = spark.table("orders")
    payments = spark.table("payments")

    # Only successful payments count towards a paid order
    successful_payments = payments.filter(F.col("status") == "SUCCESS")

    # Left join so that customers without any paid orders are still included
    orders_with_payments = orders.join(successful_payments, on="order_id", how="left")

    # Aggregate per customer
    result = orders_with_payments.groupBy("customer_id").agg(
        F.countDistinct(
            F.when(F.col("payment_id").isNotNull(), F.col("order_id"))
        ).alias("paid_orders"),
        F.coalesce(
            F.sum(F.when(F.col("payment_id").isNotNull(), F.col("order_amount"))),
            F.lit(0),
        )
        .cast("decimal(12,2)")
        .alias("revenue"),
    )

    return result


result = solve(spark)
result.show()

In [ ]:
check(result)